# config

In [ ]:
config = {
    "file_config": {
        "input_path": [
            "s3://web-parse-hw60p/linfeng/agentic_search/oa_chunk_embedding/v4/data/",
        ],
        "output_path": "s3://data-milvus/user_data/renpengli/agentic_search/bulk_parquet_data_v1",
    },
    "db_config": {
        "milvus": {
            "uri": "http://10.12.105.139:30005",
            "username": "",
            "password": "",
            "apikey": "",
            "token": "",
        }
    },
    "custom_config": {
        "spark_profile": "test",  # 测试改 test，全量改 prod
        "input_summary_path": "s3://web-parse-hw60p/linfeng/agentic_search/oa_chunk_embedding/v4/_summary.json",
        "milvus_db_name": "mineru_core",
        "milvus_collection": "rpl_agentic_search_vectors_test",
        "project_progress_path": "/share/renpengli/tmp/agent_search/",
        "spark_profile": "test",
        "run_id": "20260625T110128Z"  # datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    }
}


In [ ]:
# from datetime import datetime
# datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# Milvus Bulk Import Parquet for `chunk_embedding_oa_spark_v4` (100TB Scale)

这个 notebook 走的是适合超大规模数据的正式路线：

1. 用 Spark 递归读取 `chunk_embedding_oa_spark_v4` success JSON。
2. 用 Spark 原生算子完成绝大多数过滤与列规整；只有 `title` / `extra_info` 超长或含 `abstract` 的少数行才走 Python slow path。
3. 按分区并发写 Parquet 到 Milvus 可见对象存储目录。
4. 自动发现 `part-*.parquet`，按批次调用 Milvus bulk import。
5. 所有 import job 完成后，再创建 dense / sparse 索引。
6. 最后 `load` collection。

设计目标：
- 避免 `collect()` 回收 driver
- 避免逐行 UDF
- 保持与 `milvus_bulk_import_chunk_embedding_oa_spark_v4.ipynb` 相同的字段类型、字段长度与索引参数
- 适合 100TB 级别数据分区并发落盘后再由 Milvus bulk import 消费


In [ ]:
from __future__ import annotations
import os
os.environ["SPARK_USER"] = "renpengli"
os.environ.setdefault("XINGHE_CONF_DIR", "/share/renpengli/conf")

from datetime import datetime, timezone
from urllib.parse import urlparse
import hashlib
import json
import math
import os
import socket
import threading
import time
import requests
from pymilvus import DataType, MilvusClient
from pymilvus.bulk_writer import bulk_import, get_import_progress

from pyspark import SparkConf, StorageLevel
from pyspark.sql import SparkSession, Row
from pyspark.sql import Row, functions as F, types as T

from xinghe.s3 import *

# config = json_loads('${config}')
custom_config = config["custom_config"]
file_config = config["file_config"]
db_config = config["db_config"]
s3_config = config.get("s3_config", {})

milvus_config = db_config["milvus"]

# --- 输入输出数据 ---
SOURCE_ROOT = file_config["input_path"]
PARQUET_OUTPUT_S3_PREFIX = file_config["output_path"]

def get_s3_args(path, s3_config):
    bucket = split_s3_path(path)[0]
    s3_args = s3_config.get(bucket, {})
    if not s3_args:
        s3_args = get_s3_config(path)
    print(f"bucket={bucket}, s3_args={s3_args}")
    return bucket, s3_args
    
read_s3_bucket, read_s3_config = get_s3_args(SOURCE_ROOT[0], s3_config)
write_s3_bucket, write_s3_config = get_s3_args(PARQUET_OUTPUT_S3_PREFIX, s3_config)

# --- Milvus 连接配置 ---
MILVUS_URI = milvus_config["uri"]
MILVUS_USERNAME = milvus_config["username"]
MILVUS_PASSWORD = milvus_config["password"]
MILVUS_TOKEN = milvus_config["token"]
MILVUS_API_KEY = milvus_config["apikey"]
MILVUS_DB_NAME = custom_config["milvus_db_name"]
MILVUS_COLLECTION = custom_config["milvus_collection"]

# --- 导入策略 ---
RECREATE_COLLECTION = True
RECREATE_INDEXES_AFTER_IMPORT = True
TEST_SAMPLE_ROWS = None

# --- 字段类型与长度：与 Milvus 全量版保持一致 ---
MILVUS_VECTOR_DIM = 1024
MILVUS_MAX_CHUNK_ID_LEN = 512
MILVUS_MAX_DOC_ID_LEN = 512
MILVUS_MAX_TITLE_LEN = 4096
MILVUS_MAX_URL_LEN = 4096
MILVUS_MAX_CLASS_LEN = 512
MILVUS_MAX_JOB_ID_LEN = 256
MILVUS_MAX_TASK_ID_LEN = 256
MILVUS_MAX_EXTRA_INFO_LEN = 32768
MILVUS_TS_VARCHAR_MAX_LEN = 48
MILVUS_SOURCE_TYPE_MAX_LEN = 128
MILVUS_LANG_MAX_LEN = 64
MILVUS_MAX_METADATA_TYPE_LEN = 32
MILVUS_MAX_PUBLICATION_VENUE_TYPE_LEN = 64
MILVUS_MAX_PRIMARY_TOPIC_LEN = 512
MILVUS_MAX_PRIMARY_TOPIC_SUBFIELD_LEN = 512
MILVUS_MAX_PRIMARY_TOPIC_FIELD_LEN = 512
MILVUS_MAX_PRIMARY_TOPIC_DOMAIN_LEN = 512
MILVUS_MAX_PUBLICATION_VENUE_NAME_UNIFIED_LEN = 512
MILVUS_MAX_ACCESS_IS_OA_LEN = 32
MILVUS_AUTHOR_MAX_CAPACITY = 64
MILVUS_AUTHOR_ITEM_MAX_LEN = 512
EXTRA_INFO_NO_TRUNCATE_KEYS = ("url", "origin_url")
NOW_ISO_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

# --- Spark 输出到对象存储 ---
IMPORT_KEY_PREFIX = "/".join(PARQUET_OUTPUT_S3_PREFIX.split("/")[3:])
RUN_ID = custom_config["run_id"]
OUTPUT_WRITE_MODE = "append"  # "overwrite"
OUTPUT_COMPRESSION = "snappy"
ENABLE_PREVIEW_ROWS = False
OUTPUT_PARTITIONS = None
AUTO_PARTITIONS_PER_CORE = 4
MIN_OUTPUT_PARTITIONS = 512
MAX_OUTPUT_PARTITIONS = 65536
TARGET_PARQUET_FILE_SIZE_MB = 16384
MAX_RECORDS_PER_FILE = 0

# --- bulk import ---
MILVUS_BUCKET = "data-milvus"
MILVUS_ROOT_PATH = "milvus"
IMPORT_DISCOVER_GLOB = "part-*.parquet"
IMPORT_MAX_FILES_PER_JOB = 1024
REQUEST_TIMEOUT = 60
POLL_INTERVAL_SECONDS = 10
PROGRESS_LOG_INTERVAL_SECONDS = 30
IMPORT_JOB_TIMEOUT = "30m"
IMPORT_PROGRESS_FILE = os.path.abspath(f"{custom_config['project_progress_path'].rstrip('/')}/milvus_import_progress_{RUN_ID}.json")


# 小批量测试
spark_profile = custom_config["spark_profile"]
# 可选：小批量 smoke test 配置。默认关闭；如需先验证全链路，把 ENABLE_SMOKE_TEST 改为 True 后从第 1 步开始运行。
# True 时自动切到小批量测试模式；False 时保持正式大批量参数。
ENABLE_SMOKE_TEST = True if spark_profile == "test" else False
# 新输入前缀这里复用同一个根路径，并依赖 TEST_SAMPLE_ROWS 控制读取规模。
SMOKE_TEST_SOURCE_ROOT = [
    's3://web-parse-hw60p/linfeng/agentic_search/oa_chunk_embedding/v4/data/batch_id=00000001/source_type=en-paper-bjzn/lang=en/',
]
# smoke test 只取前多少行数据；用于快速验证写 Parquet、bulk import、建索引、load 全链路。
SMOKE_TEST_ROWS = 20000
# smoke test 写 Parquet 时使用的输出分区数；刻意设小，便于快速完成和检查产物。
SMOKE_TEST_OUTPUT_PARTITIONS = 16
# smoke test 每个 import job 最多带多少个文件；设小一些便于观察单批 job 行为。
SMOKE_TEST_MAX_FILES_PER_JOB = 128
# smoke test 目标 collection 后缀；避免和正式 collection 混用。
SMOKE_TEST_COLLECTION_SUFFIX = "_smoke"

if ENABLE_SMOKE_TEST:
    SOURCE_ROOT = SMOKE_TEST_SOURCE_ROOT
    TEST_SAMPLE_ROWS = SMOKE_TEST_ROWS
    OUTPUT_PARTITIONS = SMOKE_TEST_OUTPUT_PARTITIONS
    IMPORT_MAX_FILES_PER_JOB = SMOKE_TEST_MAX_FILES_PER_JOB
    RUN_ID = f"{RUN_ID}_smoke"
    MILVUS_COLLECTION = f"{MILVUS_COLLECTION}{SMOKE_TEST_COLLECTION_SUFFIX}"
    RECREATE_COLLECTION = True
    RECREATE_INDEXES_AFTER_IMPORT = True

print(json.dumps({
    "enable_smoke_test": ENABLE_SMOKE_TEST,
    "smoke_test_source_root": SMOKE_TEST_SOURCE_ROOT if ENABLE_SMOKE_TEST else None,
    "test_sample_rows": TEST_SAMPLE_ROWS,
    "output_partitions": OUTPUT_PARTITIONS,
    "import_max_files_per_job": IMPORT_MAX_FILES_PER_JOB,
    "run_id": RUN_ID,
    "import_progress_file": IMPORT_PROGRESS_FILE,
    "milvus_collection": MILVUS_COLLECTION,
}, ensure_ascii=False, indent=2))


# --- 索引配置：与 Milvus 全量索引保持一致 ---
EMBEDDING_INDEX_NAME = "idx_embedding_diskann"
EMBEDDING_INDEX_TYPE = "DISKANN"
EMBEDDING_METRIC_TYPE = "L2"
SCALAR_INVERTED_INDEX_FIELDS = [
    "title",
    "lang",
    "metadata_type",
    "publication_venue_type",
    "primary_topic",
    "primary_topic_subfield",
    "primary_topic_field",
    "primary_topic_domain",
    "publication_venue_name_unified",
    "access_is_oa",
    "author",
]
SCALAR_SORT_INDEX_FIELDS = [
    "publication_published_date",
    "publication_published_year",
    "citation_count",
    "influential_citation_count",
]


In [ ]:
spark_config = {
    # spark 资源配置
    "spark.default.parallelism": "1000",
    "spark.sql.shuffle.partitions": "1000",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "3g",
    "spark.driver.memory": "8g",
    "spark.driver.maxResultSize": "4g",
    "spark.executor.cores": "4",
    "spark.executor.instances": "20",
    "spark.kubernetes.executor.limit.cores": "8",  # spark.kubernetes.executor.limit.cores 是对cpu的限制
    # spark 其他配置
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.speculation": "true",
    "spark.executorEnv.SPARK_USER": "renpengli",
    #sparksql 配置
    "spark.sql.sources.partitionOverwriteMode":"dynamic",
    "spark.sql.parquet.compression.codec":"snappy",
    "spark.sql.files.maxRecordsPerFile": 50000,
    "spark.sql.adaptive.enabled":"true",
    "spark.sql.adaptive.coalescePartitions.enabled":"true",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes":"256MB",
    "spark.sql.autoBroadcastJoinThreshold":"-1",
    "spark.sql.adaptive.shuffle.targetPostShuffleInputSize":"67108864",
    # hadoop s3a 配置
    "spark.hadoop.fs.s3a.impl":"org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.hadoop.fs.s3a.connection.ssl.enabled": "false",
    # "spark.hadoop.fs.s3a.path.style.access": "true",
    # "spark.hadoop.fs.s3a.endpoint": read_s3_config["endpoint"],
    # "spark.hadoop.fs.s3a.access.key": read_s3_config["ak"],
    # "spark.hadoop.fs.s3a.secret.key": read_s3_config["sk"],
    # ========== 桶1：agentic_search 独立认证与endpoint ==========
    f"spark.hadoop.fs.s3a.bucket.{write_s3_bucket}.endpoint": write_s3_config["endpoint"],
    f"spark.hadoop.fs.s3a.bucket.{write_s3_bucket}.access.key": write_s3_config["ak"],
    f"spark.hadoop.fs.s3a.bucket.{write_s3_bucket}.secret.key": write_s3_config["sk"],
    f"spark.hadoop.fs.s3a.bucket.{write_s3_bucket}.path.style.access": "true",
    # ========== 桶2：lakehouse-data 另一套独立认证 ==========
    f"spark.hadoop.fs.s3a.bucket.{read_s3_bucket}.endpoint": read_s3_config["endpoint"],
    f"spark.hadoop.fs.s3a.bucket.{read_s3_bucket}.access.key": read_s3_config["ak"],
    f"spark.hadoop.fs.s3a.bucket.{read_s3_bucket}.secret.key": read_s3_config["sk"],
    f"spark.hadoop.fs.s3a.bucket.{read_s3_bucket}.path.style.access": "true",

    "spark.hadoop.fs.s3a.multiobjectdelete.enable":"false",
    "spark.hadoop.fs.s3a.directory.marker.retention":"keep",
    "spark.hadoop.fs.s3a.fast.upload":"true",
    "spark.hadoop.fs.s3a.connection.maximum":"1000",
    "spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version":"2",
    # kubernetes 配置
    "spark.kubernetes.executor.deleteOnTermination":"false",
    "spark.kubernetes.namespace": "dataops-paas",
    "spark.kubernetes.authenticate.driver.serviceAccountName": "spark-default",
    "spark.kubernetes.container.image.pullPolicy": "Always",
    "spark.kubernetes.container.image": "registry.sensetime.com/hadoop/dataops/apache/spark:3.5.7-data-platform",
    # event log 配置
    "spark.eventLog.enabled": "true",
    "spark.eventLog.dir": "/spark/eventLog",
    # Magic Committer（推荐 - 不需要临时目录，性能最佳）
    # 参考: https://hadoop.apache.org/docs/current/hadoop-aws/tools/hadoop-aws/committers.html
    "spark.jars":"/share/spark-jars/spark-hadoop-cloud_2.12-3.5.7.jar",
    "spark.hadoop.fs.s3a.committer.name": "magic",
    "spark.hadoop.fs.s3a.committer.magic.enabled": "true" ,
    "spark.hadoop.mapreduce.outputcommitter.factory.scheme.s3a": "org.apache.hadoop.fs.s3a.commit.S3ACommitterFactory",
    "spark.sql.sources.commitProtocolClass": "org.apache.spark.internal.io.cloud.PathOutputCommitProtocol",
    "spark.sql.parquet.output.committer.class": "org.apache.spark.internal.io.cloud.BindingParquetOutputCommitter",
    "spark.kubernetes.executor.podTemplateFile": "/share/renpengli/pod-template/pod-template-dolphin.yaml",
    "spark.kubernetes.executor.label.queue": "root.datacenter.data-producer.default",
    "spark.driver.host": os.environ.get("POD_IP"),
    "spark.submit.deployMode": "client",
}

conf = SparkConf() 
conf.setAll(list(spark_config.items()))

# 初始化Spark
master = "k8s://https://{kubernetes_service_host}:{kubernetes_service_port}".format(kubernetes_service_host = os.environ.get("KUBERNETES_SERVICE_HOST"), 
        kubernetes_service_port = os.environ.get("KUBERNETES_SERVICE_PORT"))
tt = int(time.time())
spark = SparkSession.builder.master(master).config(conf=conf).appName(f"embedding_to_milvus_{tt}").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

SOURCE_JSON_SCHEMA = T.StructType([
    T.StructField("chunk_id", T.StringType(), True),
    T.StructField("doc_id", T.StringType(), True),
    T.StructField("title", T.StringType(), True),
    T.StructField("url", T.StringType(), True),
    T.StructField("embedding", T.ArrayType(T.DoubleType(), True), True),
    T.StructField("offset", T.LongType(), True),
    T.StructField("page_no", T.LongType(), True),
    T.StructField("chunk_no", T.LongType(), True),
    T.StructField("source_type", T.StringType(), True),
    T.StructField("lang", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("job_id", T.StringType(), True),
    T.StructField("task_id", T.StringType(), True),
    T.StructField("publication_published_date", T.LongType(), True),
    T.StructField("publication_published_year", T.LongType(), True),
    T.StructField("metadata_type", T.StringType(), True),
    T.StructField("publication_venue_type", T.StringType(), True),
    T.StructField("primary_topic", T.StringType(), True),
    T.StructField("primary_topic_subfield", T.StringType(), True),
    T.StructField("primary_topic_field", T.StringType(), True),
    T.StructField("primary_topic_domain", T.StringType(), True),
    T.StructField("author", T.ArrayType(T.StringType(), True), True),
    T.StructField("publication_venue_name_unified", T.StringType(), True),
    T.StructField("access_is_oa", T.StringType(), True),
    T.StructField("citation_count", T.LongType(), True),
    T.StructField("influential_citation_count", T.LongType(), True),
    T.StructField("extra_info", T.StringType(), True),
    T.StructField("created_time", T.StringType(), True),
    T.StructField("updated_time", T.StringType(), True),
])

EMPTY_AUTHOR_ARRAY = F.from_json(
    F.lit("[]"),
    T.ArrayType(T.StringType(), True),
)

NORMALIZED_OUTPUT_SCHEMA = T.StructType([
    T.StructField("chunk_id", T.StringType(), False),
    T.StructField("doc_id", T.StringType(), False),
    T.StructField("title", T.StringType(), False),
    T.StructField("url", T.StringType(), False),
    T.StructField("embedding", T.ArrayType(T.DoubleType(), True), False),
    T.StructField("offset", T.LongType(), False),
    T.StructField("page_no", T.LongType(), False),
    T.StructField("chunk_no", T.LongType(), False),
    T.StructField("source_type", T.StringType(), False),
    T.StructField("lang", T.StringType(), False),
    T.StructField("category", T.StringType(), False),
    T.StructField("job_id", T.StringType(), False),
    T.StructField("task_id", T.StringType(), False),
    T.StructField("publication_published_date", T.LongType(), False),
    T.StructField("publication_published_year", T.LongType(), False),
    T.StructField("metadata_type", T.StringType(), False),
    T.StructField("publication_venue_type", T.StringType(), False),
    T.StructField("primary_topic", T.StringType(), False),
    T.StructField("primary_topic_subfield", T.StringType(), False),
    T.StructField("primary_topic_field", T.StringType(), False),
    T.StructField("primary_topic_domain", T.StringType(), False),
    T.StructField("author", T.ArrayType(T.StringType(), True), False),
    T.StructField("publication_venue_name_unified", T.StringType(), False),
    T.StructField("access_is_oa", T.StringType(), False),
    T.StructField("citation_count", T.LongType(), False),
    T.StructField("influential_citation_count", T.LongType(), False),
    T.StructField("extra_info", T.StringType(), False),
    T.StructField("created_time", T.StringType(), False),
    T.StructField("updated_time", T.StringType(), False)
])

spark


# funcs

In [ ]:
def ensure_s3a_path(path_str: str) -> str:
    if str(path_str).startswith("s3://"):
        return "s3a://" + str(path_str)[len("s3://"):]
    return str(path_str)


def validate_milvus_uri() -> None:
    if not MILVUS_URI.startswith("http://") and not MILVUS_URI.startswith("https://"):
        raise ValueError(f"MILVUS_URI must start with http:// or https://, got: {MILVUS_URI}")
    host_port = MILVUS_URI.split("://", 1)[1].split("/", 1)[0]
    host = host_port.split(":", 1)[0]
    port = int(host_port.split(":", 1)[1]) if ":" in host_port else 19530
    socket.getaddrinfo(host, port, socket.AF_UNSPEC, socket.SOCK_STREAM)


def build_token() -> str:
    if MILVUS_TOKEN.strip():
        return MILVUS_TOKEN.strip()
    if MILVUS_USERNAME.strip() or MILVUS_PASSWORD:
        return f"{MILVUS_USERNAME.strip()}:{MILVUS_PASSWORD}"
    return ""


def build_milvus_client() -> MilvusClient:
    kwargs = {"uri": MILVUS_URI}
    token = build_token()
    if token:
        kwargs["token"] = token
    if MILVUS_DB_NAME.strip():
        kwargs["db_name"] = MILVUS_DB_NAME.strip()
    return MilvusClient(**kwargs)


def auth_headers(*, request_timeout_header: str | None = None) -> dict:
    headers = {"Content-Type": "application/json"}
    token = build_token()
    if token:
        headers["Authorization"] = f"Bearer {token}"
    if request_timeout_header is not None and str(request_timeout_header).strip() != "":
        headers["Request-Timeout"] = str(request_timeout_header).strip()
    return headers


def compact_json(obj) -> str:
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))


def log_progress(event: str, **kwargs) -> None:
    payload = {
        "ts_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "event": event,
        **kwargs,
    }
    print(json.dumps(payload, ensure_ascii=False))


def run_with_heartbeat(label: str, fn, *, context_fn=None, interval_seconds: int | None = None):
    interval = int(interval_seconds or PROGRESS_LOG_INTERVAL_SECONDS)
    started = time.perf_counter()
    stop_event = threading.Event()

    def build_payload() -> dict:
        payload = {
            "label": label,
            "elapsed_seconds": round(time.perf_counter() - started, 1),
        }
        if context_fn is not None:
            extra = context_fn() or {}
            payload.update(extra)
        return payload

    def reporter() -> None:
        tick = 0
        while not stop_event.wait(interval):
            tick += 1
            payload = build_payload()
            payload["tick"] = tick
            log_progress("heartbeat", **payload)

    start_payload = {"label": label}
    if context_fn is not None:
        start_payload.update(context_fn() or {})
    log_progress("start", **start_payload)

    reporter_thread = None
    if interval > 0:
        reporter_thread = threading.Thread(target=reporter, daemon=True)
        reporter_thread.start()

    try:
        result = fn()
    except Exception as exc:
        stop_event.set()
        if reporter_thread is not None:
            reporter_thread.join(timeout=1)
        error_payload = build_payload()
        error_payload["error"] = str(exc)
        log_progress("error", **error_payload)
        raise

    stop_event.set()
    if reporter_thread is not None:
        reporter_thread.join(timeout=1)
    log_progress("finish", **build_payload())
    return result


def utf8_len(value: str) -> int:
    return len(str(value).encode("utf-8"))


def utf8_trim(value: str, max_bytes: int) -> str:
    if max_bytes <= 0:
        return ""
    text = value if isinstance(value, str) else str(value)
    data = text.encode("utf-8")
    if len(data) <= max_bytes:
        return text
    return data[:max_bytes].decode("utf-8", errors="ignore")


def stringify_value(value) -> str:
    if value is None:
        return ""
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, str):
        return value
    return str(value)


def normalize_object_key(value: str) -> str:
    return "/".join(part for part in str(value).split("/") if part)


def normalize_s3_prefix(value: str) -> str:
    parsed = urlparse(str(value))
    if parsed.scheme != "s3" or not parsed.netloc:
        raise ValueError(f"expected s3:// prefix, got: {value!r}")
    key_prefix = normalize_object_key(parsed.path)
    if key_prefix:
        return f"s3://{parsed.netloc}/{key_prefix}"
    return f"s3://{parsed.netloc}"


def join_object_key(prefix: str, *parts: str) -> str:
    items = []
    normalized_prefix = normalize_object_key(prefix)
    if normalized_prefix:
        items.append(normalized_prefix)
    for part in parts:
        normalized_part = normalize_object_key(part)
        if normalized_part:
            items.append(normalized_part)
    return "/".join(items)


def join_s3_uri(prefix: str, *parts: str) -> str:
    base = normalize_s3_prefix(prefix)
    suffix = join_object_key("", *parts)
    if not suffix:
        return base
    return f"{base}/{suffix}"


def is_no_truncate_extra_info_key(key: str) -> bool:
    return str(key).strip() in EXTRA_INFO_NO_TRUNCATE_KEYS


def normalize_string_field(value, max_bytes: int, *, lowercase: bool = False, default: str = "") -> str:
    text = stringify_value(value) if value is not None else default
    if value is None:
        text = default
    if lowercase:
        text = text.lower()
    if utf8_len(text) <= max_bytes:
        return text
    return utf8_trim(text, max_bytes)


def normalize_title(value):
    return normalize_string_field(value, MILVUS_MAX_TITLE_LEN, lowercase=True)


def normalize_int(value, default: int = 0) -> int:
    if value is None:
        return default
    try:
        if isinstance(value, bool):
            return int(value)
        text = str(value).strip()
        if not text or text.lower() in {"null", "none", "nan"}:
            return default
        return int(float(text))
    except Exception:
        return default


def normalize_author_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        raw_items = value
    elif isinstance(value, str):
        stripped = value.strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = json.loads(stripped)
                raw_items = parsed if isinstance(parsed, list) else [value]
            except Exception:
                raw_items = [value]
        elif stripped:
            raw_items = [value]
        else:
            raw_items = []
    else:
        raw_items = [value]
    out = []
    for item in raw_items:
        text = normalize_string_field(item, MILVUS_AUTHOR_ITEM_MAX_LEN, lowercase=True)
        if not text:
            continue
        out.append(text)
        if len(out) >= MILVUS_AUTHOR_MAX_CAPACITY:
            break
    return out


def parse_extra_info_dict(raw):
    if raw is None:
        return {}
    if isinstance(raw, dict):
        return {str(key).strip(): value for key, value in raw.items() if str(key).strip()}
    text = raw if isinstance(raw, str) else str(raw)
    text = text.strip()
    if not text:
        return {}
    try:
        obj = json.loads(text)
    except Exception:
        return {}
    if not isinstance(obj, dict):
        return {}
    return {str(key).strip(): value for key, value in obj.items() if str(key).strip()}


def truncate_extra_info_value(value, max_bytes: int):
    if value is None or isinstance(value, (bool, int, float)):
        return value
    if isinstance(value, str):
        return utf8_trim(value.strip(), max_bytes)
    if isinstance(value, dict):
        items = [(str(key).strip(), child) for key, child in value.items() if str(key).strip()]
        if not items:
            return {}
        child_budget = max(1, max_bytes // max(1, len(items)))
        return {key: truncate_extra_info_value(child, child_budget) for key, child in items}
    if isinstance(value, (list, tuple, set)):
        seq = list(value)
        if not seq:
            return []
        child_budget = max(1, max_bytes // max(1, len(seq)))
        return [truncate_extra_info_value(child, child_budget) for child in seq]
    return utf8_trim(str(value).strip(), max_bytes)


def normalize_extra_info(raw):
    if raw is None:
        return "{}"
    if isinstance(raw, str):
        text = raw.strip()
        if not text:
            return "{}"
        if text.startswith("{") and text.endswith("}") and '"abstract"' not in text and utf8_len(text) <= MILVUS_MAX_EXTRA_INFO_LEN:
            return text
    candidate = parse_extra_info_dict(raw)
    candidate.pop("abstract", None)
    payload = compact_json(candidate)
    if utf8_len(payload) <= MILVUS_MAX_EXTRA_INFO_LEN:
        return payload
    while utf8_len(payload) > MILVUS_MAX_EXTRA_INFO_LEN:
        changed = False
        truncatable_keys = [
            key
            for key in sorted(candidate, key=lambda item: utf8_len(compact_json(candidate[item])), reverse=True)
            if not is_no_truncate_extra_info_key(key)
        ]
        for key in truncatable_keys:
            current_value = candidate.get(key)
            current_len = utf8_len(compact_json(current_value))
            if current_len <= 2:
                continue
            overflow = utf8_len(payload) - MILVUS_MAX_EXTRA_INFO_LEN
            next_budget = max(1, current_len - overflow)
            new_value = truncate_extra_info_value(current_value, next_budget)
            if compact_json(new_value) == compact_json(current_value):
                continue
            candidate[key] = new_value
            payload = compact_json(candidate)
            changed = True
            if utf8_len(payload) <= MILVUS_MAX_EXTRA_INFO_LEN:
                return payload
        if changed:
            continue
        if not truncatable_keys:
            return payload
        candidate.pop(truncatable_keys[0], None)
        payload = compact_json(candidate)
    return payload


def normalize_slow_partition(rows):
    for row in rows:
        data = row.asDict(recursive=True)
        yield Row(
            chunk_id=normalize_string_field(data.get("chunk_id"), MILVUS_MAX_CHUNK_ID_LEN),
            doc_id=normalize_string_field(data.get("doc_id"), MILVUS_MAX_DOC_ID_LEN),
            title=normalize_title(data.get("title")),
            url=normalize_string_field(data.get("url"), MILVUS_MAX_URL_LEN),
            embedding=data.get("embedding") or [0.0] * MILVUS_VECTOR_DIM,
            offset=int(data.get("offset") or 0),
            page_no=int(data.get("page_no") or 0),
            chunk_no=int(data.get("chunk_no") or 0),
            source_type=normalize_string_field(data.get("source_type"), MILVUS_SOURCE_TYPE_MAX_LEN),
            lang=normalize_string_field(data.get("lang"), MILVUS_LANG_MAX_LEN, default="unknown"),
            category=normalize_string_field(data.get("category"), MILVUS_MAX_CLASS_LEN),
            job_id=normalize_string_field(data.get("job_id"), MILVUS_MAX_JOB_ID_LEN),
            task_id=normalize_string_field(data.get("task_id"), MILVUS_MAX_TASK_ID_LEN),
            publication_published_date=normalize_int(data.get("publication_published_date"), 0),
            publication_published_year=normalize_int(data.get("publication_published_year"), 0),
            metadata_type=normalize_string_field(data.get("metadata_type"), MILVUS_MAX_METADATA_TYPE_LEN),
            publication_venue_type=normalize_string_field(data.get("publication_venue_type"), MILVUS_MAX_PUBLICATION_VENUE_TYPE_LEN, lowercase=True),
            primary_topic=normalize_string_field(data.get("primary_topic"), MILVUS_MAX_PRIMARY_TOPIC_LEN, lowercase=True),
            primary_topic_subfield=normalize_string_field(data.get("primary_topic_subfield"), MILVUS_MAX_PRIMARY_TOPIC_SUBFIELD_LEN, lowercase=True),
            primary_topic_field=normalize_string_field(data.get("primary_topic_field"), MILVUS_MAX_PRIMARY_TOPIC_FIELD_LEN, lowercase=True),
            primary_topic_domain=normalize_string_field(data.get("primary_topic_domain"), MILVUS_MAX_PRIMARY_TOPIC_DOMAIN_LEN, lowercase=True),
            author=normalize_author_list(data.get("author")),
            publication_venue_name_unified=normalize_string_field(data.get("publication_venue_name_unified"), MILVUS_MAX_PUBLICATION_VENUE_NAME_UNIFIED_LEN, lowercase=True),
            access_is_oa=normalize_string_field(data.get("access_is_oa"), MILVUS_MAX_ACCESS_IS_OA_LEN),
            citation_count=normalize_int(data.get("citation_count"), -1),
            influential_citation_count=normalize_int(data.get("influential_citation_count"), -1),
            extra_info=normalize_extra_info(data.get("extra_info")),
            created_time=normalize_string_field(data.get("created_time") or NOW_ISO_UTC, MILVUS_TS_VARCHAR_MAX_LEN),
            updated_time=normalize_string_field(data.get("updated_time") or NOW_ISO_UTC, MILVUS_TS_VARCHAR_MAX_LEN),
        )


def get_content_summary_bytes(path_str: str) -> int:
    path = spark._jvm.org.apache.hadoop.fs.Path(ensure_s3a_path(path_str))
    fs = path.getFileSystem(spark._jsc.hadoopConfiguration())
    return int(fs.getContentSummary(path).getLength())


def get_total_source_bytes(source_roots: list[str]) -> int:
    total = 0
    for root in source_roots:
        total += get_content_summary_bytes(root)
    return total


def verify_output_path_writable(output_path: str) -> dict:
    target_path = ensure_s3a_path(output_path)
    jvm = spark._jvm
    conf = spark._jsc.hadoopConfiguration()
    path = jvm.org.apache.hadoop.fs.Path(target_path)
    fs = path.getFileSystem(conf)

    parent = path.getParent()
    parent_exists = False
    if parent is not None:
        parent_exists = bool(fs.exists(parent))

    target_exists = bool(fs.exists(path))

    probe_name = f"_codex_write_probe_{int(time.time())}.tmp"
    probe_path = jvm.org.apache.hadoop.fs.Path(path, probe_name)
    out = fs.create(probe_path, True)
    try:
        out.write(bytearray(b"ok"))
    finally:
        out.close()

    probe_exists = bool(fs.exists(probe_path))
    fs.delete(probe_path, False)

    return {
        "output_path": output_path,
        "output_path_s3a": target_path,
        "parent_exists": parent_exists,
        "target_exists": target_exists,
        "probe_exists": probe_exists,
    }


def resolve_target_partitions(source_bytes: int, current_partitions: int) -> int:
    default_parallelism = max(spark.sparkContext.defaultParallelism, 1)
    if OUTPUT_PARTITIONS:
        return max(1, int(OUTPUT_PARTITIONS))
    bytes_based = 0
    if source_bytes > 0 and TARGET_PARQUET_FILE_SIZE_MB > 0:
        bytes_based = math.ceil(source_bytes / (TARGET_PARQUET_FILE_SIZE_MB * 1024 * 1024))
    target = max(
        current_partitions,
        default_parallelism * AUTO_PARTITIONS_PER_CORE,
        MIN_OUTPUT_PARTITIONS,
        bytes_based,
    )
    if MAX_OUTPUT_PARTITIONS:
        target = min(target, MAX_OUTPUT_PARTITIONS)
    return max(1, int(target))


def build_output_path() -> str:
    return join_s3_uri(PARQUET_OUTPUT_S3_PREFIX, RUN_ID)


def get_import_run_prefix() -> str:
    return join_object_key(IMPORT_KEY_PREFIX, RUN_ID)


def discover_import_file_keys() -> list[str]:
    search_uri = ensure_s3a_path(join_s3_uri(f"s3://{MILVUS_BUCKET}", get_import_run_prefix(), IMPORT_DISCOVER_GLOB))
    path = spark._jvm.org.apache.hadoop.fs.Path(search_uri)
    fs = path.getFileSystem(spark._jsc.hadoopConfiguration())
    statuses = fs.globStatus(path)
    if statuses is None:
        return []
    bucket_prefix = f"/{MILVUS_BUCKET}/"
    results = []
    for status in statuses:
        if not status.isFile():
            continue
        uri_path = status.getPath().toUri().getPath()
        if uri_path.startswith(bucket_prefix):
            results.append(uri_path[len(bucket_prefix):])
        else:
            results.append(normalize_object_key(uri_path))
    return sorted(set(results))


def build_bulk_import_files_arg(import_keys: list[str]) -> list[list[str]]:
    return [[str(key).strip().lstrip("/")] for key in import_keys if str(key).strip()]


def chunk_list(items: list, size: int) -> list[list]:
    if size <= 0:
        return [items]
    return [items[idx: idx + size] for idx in range(0, len(items), size)]


def build_import_file_batches(import_keys: list[str]) -> list[list[list[str]]]:
    files_arg = build_bulk_import_files_arg(import_keys)
    return chunk_list(files_arg, IMPORT_MAX_FILES_PER_JOB)


def create_collection_schema(client: MilvusClient):
    schema = client.create_schema(auto_id=False, enable_dynamic_field=False)
    schema.add_field("chunk_id", DataType.VARCHAR, is_primary=True, max_length=MILVUS_MAX_CHUNK_ID_LEN)
    schema.add_field("doc_id", DataType.VARCHAR, max_length=MILVUS_MAX_DOC_ID_LEN)
    schema.add_field("title", DataType.VARCHAR, max_length=MILVUS_MAX_TITLE_LEN)
    schema.add_field("url", DataType.VARCHAR, max_length=MILVUS_MAX_URL_LEN)
    schema.add_field("embedding", DataType.FLOAT_VECTOR, dim=MILVUS_VECTOR_DIM)
    schema.add_field("offset", DataType.INT64)
    schema.add_field("page_no", DataType.INT64)
    schema.add_field("chunk_no", DataType.INT64)
    schema.add_field("source_type", DataType.VARCHAR, max_length=MILVUS_SOURCE_TYPE_MAX_LEN)
    schema.add_field("lang", DataType.VARCHAR, max_length=MILVUS_LANG_MAX_LEN)
    schema.add_field("category", DataType.VARCHAR, max_length=MILVUS_MAX_CLASS_LEN)
    schema.add_field("job_id", DataType.VARCHAR, max_length=MILVUS_MAX_JOB_ID_LEN)
    schema.add_field("task_id", DataType.VARCHAR, max_length=MILVUS_MAX_TASK_ID_LEN)
    schema.add_field("publication_published_date", DataType.INT64)
    schema.add_field("publication_published_year", DataType.INT64)
    schema.add_field("metadata_type", DataType.VARCHAR, max_length=MILVUS_MAX_METADATA_TYPE_LEN)
    schema.add_field("publication_venue_type", DataType.VARCHAR, max_length=MILVUS_MAX_PUBLICATION_VENUE_TYPE_LEN)
    schema.add_field("primary_topic", DataType.VARCHAR, max_length=MILVUS_MAX_PRIMARY_TOPIC_LEN)
    schema.add_field("primary_topic_subfield", DataType.VARCHAR, max_length=MILVUS_MAX_PRIMARY_TOPIC_SUBFIELD_LEN)
    schema.add_field("primary_topic_field", DataType.VARCHAR, max_length=MILVUS_MAX_PRIMARY_TOPIC_FIELD_LEN)
    schema.add_field("primary_topic_domain", DataType.VARCHAR, max_length=MILVUS_MAX_PRIMARY_TOPIC_DOMAIN_LEN)
    schema.add_field(
        "author",
        DataType.ARRAY,
        element_type=DataType.VARCHAR,
        max_capacity=MILVUS_AUTHOR_MAX_CAPACITY,
        max_length=MILVUS_AUTHOR_ITEM_MAX_LEN,
    )
    schema.add_field("publication_venue_name_unified", DataType.VARCHAR, max_length=MILVUS_MAX_PUBLICATION_VENUE_NAME_UNIFIED_LEN)
    schema.add_field("access_is_oa", DataType.VARCHAR, max_length=MILVUS_MAX_ACCESS_IS_OA_LEN)
    schema.add_field("citation_count", DataType.INT64)
    schema.add_field("influential_citation_count", DataType.INT64)
    schema.add_field("extra_info", DataType.VARCHAR, max_length=MILVUS_MAX_EXTRA_INFO_LEN)
    schema.add_field("created_time", DataType.VARCHAR, max_length=MILVUS_TS_VARCHAR_MAX_LEN)
    schema.add_field("updated_time", DataType.VARCHAR, max_length=MILVUS_TS_VARCHAR_MAX_LEN)
    return schema


def prepare_collection() -> dict:
    client = build_milvus_client()
    validate_milvus_uri()
    exists = client.has_collection(MILVUS_COLLECTION)
    if exists and RECREATE_COLLECTION:
        print(f"dropping collection: {MILVUS_COLLECTION}")
        started = time.perf_counter()
        client.drop_collection(MILVUS_COLLECTION)
        print(f"collection dropped in {time.perf_counter() - started:.2f}s")
        exists = False
    if not exists:
        print(f"creating collection: {MILVUS_COLLECTION}")
        started = time.perf_counter()
        schema = create_collection_schema(client)
        client.create_collection(collection_name=MILVUS_COLLECTION, schema=schema)
        print(f"collection created in {time.perf_counter() - started:.2f}s")
        return {"created": True, "reused": False}
    return {"created": False, "reused": True}


def wait_import_job(job_id: str) -> dict:
    while True:
        progress_resp = get_import_progress(
            url=MILVUS_URI,
            job_id=str(job_id),
            api_key=MILVUS_API_KEY,
        )
        progress_resp.raise_for_status()
        payload = progress_resp.json()
        # 故意不打印每次轮询的原始 progress payload，避免第 5 步刷屏。
        state = payload.get("data", {}).get("state")
        if state in {"Completed", "Failed"}:
            return payload
        time.sleep(POLL_INTERVAL_SECONDS)


def build_probe_index_params():
    index_params = MilvusClient.prepare_index_params()
    index_params.add_index(
        field_name="embedding",
        index_name=EMBEDDING_INDEX_NAME,
        index_type=EMBEDDING_INDEX_TYPE,
        metric_type=EMBEDDING_METRIC_TYPE,
    )
    for field_name in SCALAR_INVERTED_INDEX_FIELDS:
        index_params.add_index(
            field_name=field_name,
            index_name=f"idx_{field_name}_inverted",
            index_type="INVERTED",
        )
    for field_name in SCALAR_SORT_INDEX_FIELDS:
        index_params.add_index(
            field_name=field_name,
            index_name=f"idx_{field_name}_sort",
            index_type="STL_SORT",
        )
    return index_params


def create_collection_indexes() -> dict:
    client = build_milvus_client()
    validate_milvus_uri()
    index_params = build_probe_index_params()
    existing = set(client.list_indexes(MILVUS_COLLECTION))
    created = []
    recreated = []
    for index_param in index_params:
        if index_param.index_name in existing and RECREATE_INDEXES_AFTER_IMPORT:
            print(f"dropping existing index: {index_param.index_name}")
            client.drop_index(MILVUS_COLLECTION, index_param.index_name)
            recreated.append(index_param.index_name)
        if index_param.index_name not in existing or RECREATE_INDEXES_AFTER_IMPORT:
            print(f"creating index {index_param.index_name} on field {index_param.field_name}")
            single = MilvusClient.prepare_index_params()
            single.add_index(**index_param.to_dict())
            client.create_index(MILVUS_COLLECTION, single)
            created.append(index_param.to_dict())
    return {
        "collection": MILVUS_COLLECTION,
        "index_count": len(created),
        "recreated_index_names": recreated,
        "indexes": created,
    }


def load_collection() -> dict:
    client = build_milvus_client()
    validate_milvus_uri()
    client.load_collection(MILVUS_COLLECTION)
    load_state = client.get_load_state(MILVUS_COLLECTION)
    return {
        "collection": MILVUS_COLLECTION,
        "load_state": str(load_state.get("state")),
        **({"load_progress": load_state.get("progress")} if "progress" in load_state else {}),
    }

def read_source_root(root_path: str):
    return (
        spark.read
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", "part-*.jsonl")
        .text(root_path)
    )

def utf8_length_ok(column_name: str, max_len: int):
    return F.length(F.encode(F.col(column_name), "UTF-8")) <= F.lit(max_len)


# 第 5 步：按 batch 断点续跑；已完成 batch 直接跳过，只重试失败批次。
def utcnow_text() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def build_import_batch_signature(files_batch: list[list[str]]) -> str:
    normalized = [
        [str(item).strip().lstrip("/") for item in group if str(item).strip()]
        for group in files_batch
    ]
    payload = json.dumps(normalized, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()


def load_import_checkpoint(path: str) -> dict:
    if not path or not os.path.exists(path):
        return {}
    with open(path, "r", encoding="utf-8") as fp:
        payload = json.load(fp)
    if not isinstance(payload, dict):
        raise RuntimeError(f"invalid import checkpoint payload in {path}")
    return payload


def save_import_checkpoint(path: str, checkpoint_batches: dict, *, batch_count: int) -> None:
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    payload = {
        "run_id": RUN_ID,
        "collection": MILVUS_COLLECTION,
        "db_name": MILVUS_DB_NAME,
        "import_progress_file": path,
        "batch_count": batch_count,
        "updated_at_utc": utcnow_text(),
        "batches": [checkpoint_batches[key] for key in sorted(checkpoint_batches)],
    }
    tmp_path = f"{path}.tmp"
    with open(tmp_path, "w", encoding="utf-8") as fp:
        json.dump(payload, fp, ensure_ascii=False, indent=2)
        fp.write("\n")
    os.replace(tmp_path, path)



# main progress

In [ ]:
# 第 1 步：创建或复用目标 collection；这一步只建 schema，不建索引。
collection_started = time.perf_counter()
collection_result = run_with_heartbeat(
    "prepare_collection",
    prepare_collection,
    context_fn=lambda: {"collection": MILVUS_COLLECTION, "db_name": MILVUS_DB_NAME},
)
print(json.dumps({
    "collection": MILVUS_COLLECTION,
    **collection_result,
    "elapsed_seconds": round(time.perf_counter() - collection_started, 2),
}, ensure_ascii=False, indent=2))


In [ ]:
source_roots = [ensure_s3a_path(str(x)) for x in SOURCE_ROOT if str(x).strip()]

In [ ]:
# 第 2 步：读取源数据并准备写 Parquet；绝大多数行走 Spark 原生 fast path，只有超长字段或 extra_info 需要裁剪的少数行走 Python slow path。
if not source_roots:
    raise RuntimeError("SOURCE_ROOT must not be empty")

log_progress("info", label="prepare_data_plan", run_id=RUN_ID, sample_mode=bool(TEST_SAMPLE_ROWS), source_root_count=len(source_roots), source_roots=source_roots)




raw_df = None
for root_path in source_roots:
    next_df = read_source_root(root_path)
    raw_df = next_df if raw_df is None else raw_df.unionByName(next_df)

parsed_df = (
    raw_df
    .select(F.from_json(F.col("value"), SOURCE_JSON_SCHEMA).alias("d"))
    .select("d.*")
)

author_expr = F.filter(
    F.transform(
        F.coalesce(F.col("author"), EMPTY_AUTHOR_ARRAY),
        lambda x: F.lower(F.coalesce(x, F.lit(""))),
    ),
    lambda x: x != "",
)

base_df = (
    parsed_df
    .where(F.col("chunk_id").isNotNull() & F.col("embedding").isNotNull())
    .withColumn("embedding_dim", F.size("embedding"))
    .where(F.col("embedding_dim") == F.lit(MILVUS_VECTOR_DIM))
    .drop("embedding_dim")
    .select(
        F.coalesce(F.col("chunk_id"), F.lit("")) .alias("chunk_id"),
        F.coalesce(F.col("doc_id"), F.lit("")) .alias("doc_id"),
        F.lower(F.coalesce(F.col("title"), F.lit(""))).alias("title"),
        F.coalesce(F.col("url"), F.lit("")) .alias("url"),
        F.col("embedding"),
        F.coalesce(F.col("offset"), F.lit(0)).cast("long").alias("offset"),
        F.coalesce(F.col("page_no"), F.lit(0)).cast("long").alias("page_no"),
        F.coalesce(F.col("chunk_no"), F.lit(0)).cast("long").alias("chunk_no"),
        F.coalesce(F.col("source_type"), F.lit("")) .alias("source_type"),
        F.coalesce(F.col("lang"), F.lit("unknown")) .alias("lang"),
        F.coalesce(F.col("category"), F.lit("")) .alias("category"),
        F.coalesce(F.col("job_id"), F.lit("")) .alias("job_id"),
        F.coalesce(F.col("task_id"), F.lit("")) .alias("task_id"),
        F.coalesce(F.col("publication_published_date"), F.lit(0)).cast("long").alias("publication_published_date"),
        F.coalesce(F.col("publication_published_year"), F.lit(0)).cast("long").alias("publication_published_year"),
        F.coalesce(F.col("metadata_type"), F.lit("")) .alias("metadata_type"),
        F.lower(F.coalesce(F.col("publication_venue_type"), F.lit(""))).alias("publication_venue_type"),
        F.lower(F.coalesce(F.col("primary_topic"), F.lit(""))).alias("primary_topic"),
        F.lower(F.coalesce(F.col("primary_topic_subfield"), F.lit(""))).alias("primary_topic_subfield"),
        F.lower(F.coalesce(F.col("primary_topic_field"), F.lit(""))).alias("primary_topic_field"),
        F.lower(F.coalesce(F.col("primary_topic_domain"), F.lit(""))).alias("primary_topic_domain"),
        author_expr.alias("author"),
        F.lower(F.coalesce(F.col("publication_venue_name_unified"), F.lit(""))).alias("publication_venue_name_unified"),
        F.coalesce(F.col("access_is_oa"), F.lit("")) .alias("access_is_oa"),
        F.coalesce(F.col("citation_count"), F.lit(-1)).cast("long").alias("citation_count"),
        F.coalesce(F.col("influential_citation_count"), F.lit(-1)).cast("long").alias("influential_citation_count"),
        F.coalesce(F.col("extra_info"), F.lit("{}")).alias("extra_info"),
        F.coalesce(F.col("created_time"), F.lit(NOW_ISO_UTC)).alias("created_time"),
        F.coalesce(F.col("updated_time"), F.lit(NOW_ISO_UTC)).alias("updated_time"),
    )
)

if TEST_SAMPLE_ROWS:
    base_df = base_df.limit(TEST_SAMPLE_ROWS)


string_checks = [
    utf8_length_ok("chunk_id", MILVUS_MAX_CHUNK_ID_LEN),
    utf8_length_ok("doc_id", MILVUS_MAX_DOC_ID_LEN),
    utf8_length_ok("title", MILVUS_MAX_TITLE_LEN),
    utf8_length_ok("url", MILVUS_MAX_URL_LEN),
    utf8_length_ok("source_type", MILVUS_SOURCE_TYPE_MAX_LEN),
    utf8_length_ok("lang", MILVUS_LANG_MAX_LEN),
    utf8_length_ok("category", MILVUS_MAX_CLASS_LEN),
    utf8_length_ok("job_id", MILVUS_MAX_JOB_ID_LEN),
    utf8_length_ok("task_id", MILVUS_MAX_TASK_ID_LEN),
    utf8_length_ok("metadata_type", MILVUS_MAX_METADATA_TYPE_LEN),
    utf8_length_ok("publication_venue_type", MILVUS_MAX_PUBLICATION_VENUE_TYPE_LEN),
    utf8_length_ok("primary_topic", MILVUS_MAX_PRIMARY_TOPIC_LEN),
    utf8_length_ok("primary_topic_subfield", MILVUS_MAX_PRIMARY_TOPIC_SUBFIELD_LEN),
    utf8_length_ok("primary_topic_field", MILVUS_MAX_PRIMARY_TOPIC_FIELD_LEN),
    utf8_length_ok("primary_topic_domain", MILVUS_MAX_PRIMARY_TOPIC_DOMAIN_LEN),
    utf8_length_ok("publication_venue_name_unified", MILVUS_MAX_PUBLICATION_VENUE_NAME_UNIFIED_LEN),
    utf8_length_ok("access_is_oa", MILVUS_MAX_ACCESS_IS_OA_LEN),
    utf8_length_ok("created_time", MILVUS_TS_VARCHAR_MAX_LEN),
    utf8_length_ok("updated_time", MILVUS_TS_VARCHAR_MAX_LEN),
]
string_fields_ok = string_checks[0]
for cond in string_checks[1:]:
    string_fields_ok = string_fields_ok & cond

author_capacity_ok = F.size(F.col("author")) <= F.lit(MILVUS_AUTHOR_MAX_CAPACITY)
author_item_ok = ~F.exists(
    F.col("author"),
    lambda x: F.length(F.encode(x, "UTF-8")) > F.lit(MILVUS_AUTHOR_ITEM_MAX_LEN),
)
extra_info_ok = (
    (F.length(F.encode(F.col("extra_info"), "UTF-8")) <= F.lit(MILVUS_MAX_EXTRA_INFO_LEN))
    & (F.instr(F.col("extra_info"), '"abstract"') == 0)
)
slow_condition = (~string_fields_ok) | (~author_capacity_ok) | (~author_item_ok) | (~extra_info_ok)

fast_df = base_df.where(~slow_condition)
slow_df = base_df.where(slow_condition)
normalized_slow_df = spark.createDataFrame(slow_df.rdd.mapPartitions(normalize_slow_partition), schema=NORMALIZED_OUTPUT_SCHEMA)
normalized_df = fast_df.unionByName(normalized_slow_df)

print(json.dumps({
    "sample_mode": bool(TEST_SAMPLE_ROWS),
    "run_id": RUN_ID,
    "current_partitions": normalized_df.rdd.getNumPartitions(),
    "source_roots": source_roots,
    "output_path": build_output_path(),
}, ensure_ascii=False, indent=2))

normalized_df.printSchema()
if ENABLE_PREVIEW_ROWS:
    preview_rows = run_with_heartbeat(
        "preview_rows_collect",
        lambda: [row.asDict(recursive=True) for row in normalized_df.limit(3).collect()],
        context_fn=lambda: {"run_id": RUN_ID, "partitions": normalized_df.rdd.getNumPartitions()},
    )
    print(f"preview_row:{json.dumps(preview_rows, ensure_ascii=False, indent=2)}")
else:
    log_progress("info", label="preview_rows_skipped", run_id=RUN_ID, reason="ENABLE_PREVIEW_ROWS is False")


In [ ]:
# 第 3 步：按目标文件大小估算输出分区，并由 Spark 分区并发写 Parquet 到对象存储。
source_bytes = get_total_source_bytes(source_roots)
current_partitions = max(normalized_df.rdd.getNumPartitions(), 1)
target_partitions = resolve_target_partitions(source_bytes, current_partitions)

write_df = normalized_df
if target_partitions > current_partitions:
    write_df = normalized_df.repartition(target_partitions, F.col("chunk_id"))
elif target_partitions < current_partitions:
    write_df = normalized_df.coalesce(target_partitions)

output_path = build_output_path()
writer = (
    write_df.write
    .mode(OUTPUT_WRITE_MODE)
    .option("compression", OUTPUT_COMPRESSION)
)
if MAX_RECORDS_PER_FILE and int(MAX_RECORDS_PER_FILE) > 0:
    writer = writer.option("maxRecordsPerFile", int(MAX_RECORDS_PER_FILE))

write_started = time.perf_counter()
run_with_heartbeat(
    "write_parquet",
    lambda: writer.parquet(ensure_s3a_path(output_path)),
    context_fn=lambda: {
        "run_id": RUN_ID,
        "output_path": output_path,
        "current_partitions": current_partitions,
        "target_partitions": target_partitions,
    },
)

print(json.dumps({
    "run_id": RUN_ID,
    "source_bytes": source_bytes,
    "current_partitions": current_partitions,
    "target_partitions": target_partitions,
    "output_path": output_path,
    "output_compression": OUTPUT_COMPRESSION,
    "elapsed_seconds": round(time.perf_counter() - write_started, 2),
}, ensure_ascii=False, indent=2))


In [ ]:
# 第 4 步：自动发现本批次输出的 Parquet 文件，并按单次 job 最大文件数切成 import batches。
log_progress("info", label="discover_import_files", run_id=RUN_ID, import_run_prefix=get_import_run_prefix(), import_glob=IMPORT_DISCOVER_GLOB)
import_file_keys = discover_import_file_keys()
if not import_file_keys:
    raise RuntimeError(f"no parquet files discovered under {get_import_run_prefix()!r} matching {IMPORT_DISCOVER_GLOB!r}")

import_file_batches = build_import_file_batches(import_file_keys)
print(json.dumps({
    "run_id": RUN_ID,
    "import_run_prefix": get_import_run_prefix(),
    "import_glob": IMPORT_DISCOVER_GLOB,
    "file_count": len(import_file_keys),
    "batch_count": len(import_file_batches),
    "max_files_per_job": IMPORT_MAX_FILES_PER_JOB,
    "first_files": import_file_keys[:10],
    "first_batch_preview": import_file_batches[0][:5] if import_file_batches else [],
}, ensure_ascii=False, indent=2))


In [ ]:
# 第 5 步：按 batch 断点续跑；已完成 batch 直接跳过，只重试失败批次。
checkpoint_payload = load_import_checkpoint(IMPORT_PROGRESS_FILE)
if checkpoint_payload:
    if checkpoint_payload.get("run_id") not in {None, RUN_ID}:
        raise RuntimeError(f"checkpoint run_id mismatch: {checkpoint_payload.get('run_id')} != {RUN_ID}")
    if checkpoint_payload.get("collection") not in {None, MILVUS_COLLECTION}:
        raise RuntimeError(f"checkpoint collection mismatch: {checkpoint_payload.get('collection')} != {MILVUS_COLLECTION}")
    if checkpoint_payload.get("db_name") not in {None, MILVUS_DB_NAME}:
        raise RuntimeError(f"checkpoint db_name mismatch: {checkpoint_payload.get('db_name')} != {MILVUS_DB_NAME}")
    previous_batch_count = checkpoint_payload.get("batch_count")
    if previous_batch_count not in {None, len(import_file_batches)}:
        raise RuntimeError(f"checkpoint batch_count mismatch: {previous_batch_count} != {len(import_file_batches)}")

checkpoint_batches = {}
for item in checkpoint_payload.get("batches", []):
    if not isinstance(item, dict):
        continue
    batch_no = item.get("batch_no")
    if isinstance(batch_no, int):
        checkpoint_batches[batch_no] = item

save_import_checkpoint(IMPORT_PROGRESS_FILE, checkpoint_batches, batch_count=len(import_file_batches))

job_results = []
for batch_no, files_batch in enumerate(import_file_batches, start=1):
    batch_signature = build_import_batch_signature(files_batch)
    existing = checkpoint_batches.get(batch_no)
    if existing and existing.get("batch_signature") not in {None, batch_signature}:
        raise RuntimeError(
            f"checkpoint batch signature mismatch for batch {batch_no}; use a new RUN_ID or delete {IMPORT_PROGRESS_FILE}"
        )

    if existing and existing.get("final_state") == "Completed":
        log_progress("info", label="skip_completed_import_batch", batch_no=batch_no, job_id=existing.get("job_id"), file_count=len(files_batch))
        job_results.append(existing)
        continue

    if existing and existing.get("job_id") and existing.get("final_state") not in {"Completed", "Failed"}:
        job_id = existing["job_id"]
        log_progress("info", label="resume_inflight_import_batch", batch_no=batch_no, job_id=job_id, file_count=len(files_batch))
        final_payload = wait_import_job(job_id)
        final_state = final_payload.get("data", {}).get("state")
        batch_result = {
            **existing,
            "batch_no": batch_no,
            "batch_signature": batch_signature,
            "file_count": len(files_batch),
            "files_preview": files_batch[:5],
            "final_state": final_state,
            "payload": final_payload.get("data", {}),
            "last_finished_at_utc": utcnow_text(),
        }
        checkpoint_batches[batch_no] = batch_result
        save_import_checkpoint(IMPORT_PROGRESS_FILE, checkpoint_batches, batch_count=len(import_file_batches))
        job_results.append(batch_result)
        log_progress("info", label="import_batch_finished", batch_no=batch_no, job_id=job_id, final_state=final_state, file_count=len(files_batch), attempt=batch_result.get("attempt"))
        if final_state != "Completed":
            raise RuntimeError(f"import batch failed: {json.dumps(batch_result, ensure_ascii=False)}")
        continue

    attempt = int((existing or {}).get("attempt") or 0) + 1
    if existing and existing.get("final_state") == "Failed":
        log_progress("info", label="retry_failed_import_batch", batch_no=batch_no, previous_job_id=existing.get("job_id"), file_count=len(files_batch), attempt=attempt)
    else:
        log_progress("info", label="submit_import_batch", batch_no=batch_no, batch_count=len(import_file_batches), file_count=len(files_batch), attempt=attempt)

    try:
        resp = bulk_import(
            url=MILVUS_URI,
            collection_name=MILVUS_COLLECTION,
            db_name=MILVUS_DB_NAME,
            files=files_batch,
            api_key=MILVUS_API_KEY,
            options={"timeout": IMPORT_JOB_TIMEOUT},
        )
        resp.raise_for_status()
        create_payload = resp.json()
    except Exception as exc:
        batch_result = {
            "batch_no": batch_no,
            "batch_signature": batch_signature,
            "file_count": len(files_batch),
            "files_preview": files_batch[:5],
            "attempt": attempt,
            "final_state": "SubmitFailed",
            "error": str(exc),
            "last_finished_at_utc": utcnow_text(),
        }
        checkpoint_batches[batch_no] = batch_result
        save_import_checkpoint(IMPORT_PROGRESS_FILE, checkpoint_batches, batch_count=len(import_file_batches))
        log_progress("error", label="import_batch_submit_failed", batch_no=batch_no, file_count=len(files_batch), attempt=attempt, error=str(exc))
        raise

    job_id = create_payload["data"]["jobId"]
    submitted_result = {
        "batch_no": batch_no,
        "batch_signature": batch_signature,
        "job_id": job_id,
        "file_count": len(files_batch),
        "files_preview": files_batch[:5],
        "attempt": attempt,
        "final_state": "Submitted",
        "create_payload": create_payload.get("data", {}),
        "last_submitted_at_utc": utcnow_text(),
    }
    checkpoint_batches[batch_no] = submitted_result
    save_import_checkpoint(IMPORT_PROGRESS_FILE, checkpoint_batches, batch_count=len(import_file_batches))
    log_progress("info", label="import_job_created", batch_no=batch_no, job_id=job_id, file_count=len(files_batch), attempt=attempt)

    final_payload = wait_import_job(job_id)
    final_state = final_payload.get("data", {}).get("state")
    batch_result = {
        **submitted_result,
        "final_state": final_state,
        "payload": final_payload.get("data", {}),
        "last_finished_at_utc": utcnow_text(),
    }
    checkpoint_batches[batch_no] = batch_result
    save_import_checkpoint(IMPORT_PROGRESS_FILE, checkpoint_batches, batch_count=len(import_file_batches))
    job_results.append(batch_result)
    log_progress("info", label="import_batch_finished", batch_no=batch_no, job_id=job_id, final_state=final_state, file_count=len(files_batch), attempt=attempt)
    if final_state != "Completed":
        raise RuntimeError(f"import batch failed: {json.dumps(batch_result, ensure_ascii=False)}")

print(json.dumps({
    "import_progress_file": IMPORT_PROGRESS_FILE,
    "job_results": job_results,
}, ensure_ascii=False, indent=2))


In [ ]:
# 第 6 步：所有 import jobs 完成后创建索引。
if not job_results or any(item.get("final_state") != "Completed" for item in job_results):
    raise RuntimeError("not all import jobs completed successfully; skip index creation")

index_started = time.perf_counter()
index_result = run_with_heartbeat(
    "create_indexes",
    create_collection_indexes,
    context_fn=lambda: {"collection": MILVUS_COLLECTION},
)
print(json.dumps({
    **index_result,
    "elapsed_seconds": round(time.perf_counter() - index_started, 2),
}, ensure_ascii=False, indent=2))


In [ ]:
# 第 7 步：load collection，使其可直接用于检索。
if not job_results or any(item.get("final_state") != "Completed" for item in job_results):
    raise RuntimeError("not all import jobs completed successfully; skip load")

load_started = time.perf_counter()
load_result = run_with_heartbeat(
    "load_collection",
    load_collection,
    context_fn=lambda: {"collection": MILVUS_COLLECTION},
)
print(json.dumps({
    **load_result,
    "elapsed_seconds": round(time.perf_counter() - load_started, 2),
}, ensure_ascii=False, indent=2))
